# Multi-Scenario Policy Scenario Analytical Post-Processing
### Decision Support System Scenario Evaluation and Stress-Testing Compilation

This notebook handles all analytical post-processing for the seven isolated optimization runs. Now that the optimization processes are run independently in separate terminals or across different laptops, this notebook loads the final pickled results, hydrates their spatial graph references, and runs the downstream thesis evaluation modules:

1. **Automated Policy Topological Consistency Audit** (Jaccard and Cosine matrix alignments)
2. **Multi-Scenario Sensitivity and Pareto Frontier Analysis** (3D Pareto frontier sweeps)
3. **High-Fidelity Microscopic Validation** (dynamic agent queue simulation)
4. **Analytical Post-Processing and Telemetry Reporting**

In [ ]:
import os
import sys
import pickle
import yaml
from pathlib import Path

# Ensure the project root utilities can be imported correctly
project_root = Path(".").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from utils.city_graph import CityGraph
from utils.direct_demand_sampler import DirectDemandSampler, DDMConfig
from utils.consistency_engine import ConsistencyEngine
from utils.sensitivity_testing import SensitivitySuite
from utils.simulation import StaticSurrogateEvaluator, SimulationSetup

print("[SETUP] Core modules successfully imported.")

### Graph Reference Hydration Engine

Since chromosomes contain spatial and edge references to the `CityGraph` model, loading them from serialized `.pkl` files across different python execution environments can break object identity matching. This hydration block matches the serialized keys back to the freshly instantiated road network.

In [ ]:
def reconstruct_chromosome(chrom, city_graph):
    """
    Hydrates edge/graph references inside the serialized chromosome routes,
    allocations, and pheromone views against a freshly loaded CityGraph instance.
    This guarantees object identity alignment across multiple runs and laptops.
    """
    edge_lookup = {((e.start.lon, e.start.lat), (e.end.lon, e.end.lat)): e for e in city_graph.graph}
    
    for route in chrom.routes:
        route.cg = city_graph
        route.path = [edge_lookup[k] for k in route.path_keys if k in edge_lookup]
    
    # Align allocation dictionary keys back to hydrated Route object instances by route ID
    route_lookup = {r.id: r for r in chrom.routes}
    chrom.allocation = {route_lookup[old_route.id]: val for old_route, val in chrom.allocation.items() if old_route.id in route_lookup}
    
    if chrom.pheromones:
        from utils.pheromone import _TauView
        chrom.pheromones._edge_repr = {}
        for k in chrom.pheromones._tau.keys():
            repr_edge = edge_lookup.get(k)
            if repr_edge:
                chrom.pheromones._edge_repr[k] = repr_edge
        chrom.pheromones.tau = _TauView(chrom.pheromones._tau, chrom.pheromones._edge_repr)

print("[HYDRATOR] Reference hydration engine initialized.")

### Step 1: Pre-Flight Scenario Result Audit

This step checks each scenario's target output folder (`outputs/opt_p1/` through `outputs/opt_p7/`) for their corresponding best chromosome file. If any are missing, it stops gracefully and tells you which files need to be copied into place.

In [ ]:
SCENARIOS = {
    "P1_Scientific_Anchor": "outputs/opt_p1/P1_Scientific_Anchor_best_chromosome.pkl",
    "P2_Moderate_Fleet_Stress": "outputs/opt_p2/P2_Moderate_Fleet_Stress_best_chromosome.pkl",
    "P3_Low_Fleet_Austerity": "outputs/opt_p3/P3_Low_Fleet_Austerity_best_chromosome.pkl",
    "P4_Mid_Exploration": "outputs/opt_p4/P4_Mid_Exploration_best_chromosome.pkl",
    "P5_Radical_Exploration": "outputs/opt_p5/P5_Radical_Exploration_best_chromosome.pkl",
    "P6_Moderate_Friction": "outputs/opt_p6/P6_Moderate_Friction_best_chromosome.pkl",
    "P7_Extreme_Friction": "outputs/opt_p7/P7_Extreme_Friction_best_chromosome.pkl"
}

missing_files = []
for name, file_path in SCENARIOS.items():
    if not os.path.exists(file_path):
        missing_files.append(file_path)

if missing_files:
    print("==================================================")
    print("!!! MISSING SCENARIO RESULTS DETECTED !!!")
    print("==================================================")
    print("The following pickled chromosomes are missing:")
    for path in missing_files:
        print(f" - {path}")
    print("\nPlease ensure that all 7 optimization scripts have finished running")
    print("and their output pickle files are copied into their respective directories.")
    raise FileNotFoundError("Please copy all scenario best chromosomes to run downstream blocks.")
else:
    print("[PRE-FLIGHT] Success! All 7 optimal scenario files are present.")

### Step 2: Environment Initialization and Base Ingestion

We load the OpenStreetMap road graph and direct demand models using the calibrated baseline definitions.

In [ ]:
CONFIG_PATH = "configs/iligan_configs.yaml"
with open(CONFIG_PATH, "r") as f:
    raw_config = yaml.safe_load(f)

# 1. Ingest OpenStreetMap road graph using the strict bounding box (min_lon, min_lat, max_lon, max_lat)
bbox = (124.1500, 8.1500, 124.4000, 8.3300)
city_network = CityGraph(bbox=bbox, name="Iligan_City_Arterial", pbf_path="utils/data/iligan-city.pbf")
city_network.stitch_graph()

# 2. Initialize Direct Demand Model
ddm_cfg = DDMConfig(alpha=0.6, beta=0.4, cache_dir="utils/.cache")
demand_sampler = DirectDemandSampler(city=city_network, config=ddm_cfg, verbose=True)

print(f"\n[BASE SETUP] Graph initialized with {len(city_network.nodes)} nodes and {len(city_network.graph)} edges.")

### Step 3: Load and Hydrate Chromosomes

In [ ]:
optimized_chromosomes = []

for name, file_path in SCENARIOS.items():
    with open(file_path, "rb") as f:
        chrom = pickle.load(f)
    
    # Hydrate references back to freshly built CityGraph
    reconstruct_chromosome(chrom, city_network)
    optimized_chromosomes.append(chrom)
    print(f"Loaded & hydrated: {name} (Surrogate Cost: {chrom.cost:.4f})")

### Step 4: Automated Policy Topological Consistency Audit

Computes topological edge set similarity (Jaccard) and degree distribution alignments (Cosine) to evaluate structural divergence.

In [ ]:
audit_results = ConsistencyEngine.analyze_consistency(chromosomes=optimized_chromosomes)

jaccard_matrix = audit_results['jaccard_matrix']
cosine_matrix = audit_results['cosine_matrix']

print("--- METRIC VALIDATION REPORT ---")
print(f"Mean Topological Edge Jaccard Similarity: {audit_results['mean_jaccard']:.4f}")
print(f"Mean Junction Degree Cosine Alignment:   {audit_results['mean_cosine']:.4f}")
print(f"Consistency Success Condition Met (Jaccard >= 0.80): {audit_results['success']}")

print("\nEdge Jaccard Similarity Matrix:")
for row in jaccard_matrix:
    print("  " + " ".join(f"{val:.4f}" for val in row))
    
print("\nDegree Cosine Similarity Matrix:")
for row in cosine_matrix:
    print("  " + " ".join(f"{val:.4f}" for val in row))

if not audit_results['success']:
    print("\n[METHODOLOGICAL NOTE] Edge sets diverge topological threshold (Jaccard < 0.80).")
    print("This divergence is expected because these seven profiles represent radically different")
    print("constraints and sweeps: Resource Limits (P3), Algorithmic Drift (P5), and Commuter Friction limits (P7).")
else:
    print("\n[SUCCESS] Structural consistency verified under identical baseline environments.")

### Step 5: Multi-Scenario Sensitivity and Pareto Frontier Analysis

Runs environmental stress-testing sweeps on the Primary baseline (Scenario P1 Scientific Anchor) and generates the 3D Pareto frontier plot.

In [ ]:
primary_chromosome = optimized_chromosomes[0]

# Reconstruct high-fidelity evaluator
surrogate_evaluator = StaticSurrogateEvaluator(
    config=raw_config, 
    city_graph=city_network, 
    demand_sampler=demand_sampler, 
    num_samples=1000
)

output_plot = "outputs/plots/3d_pareto_frontier.png"
Path(output_plot).parent.mkdir(parents=True, exist_ok=True)

sensitivity_summary = SensitivitySuite.execute_sensitivity_suite(
    evaluator=surrogate_evaluator, 
    chromosome=primary_chromosome,
    output_plot_path=output_plot
)

print(f"\n[SENSITIVITY FINISHED] Sensitivity suite execution finalized.")
print(f"3D Pareto Frontier visual scatter plot saved to: {output_plot}")

### Step 6: High-Fidelity Microscopic Validation Execution

Launches the multi-agent queue simulator with strict vehicle spacing to validate optimal routes.

In [ ]:
production_routes = primary_chromosome.routes

sim_setup = SimulationSetup(city_query="IliganCity", config=raw_config, routes=production_routes)
micro_simulation = sim_setup.build()

# Force strict vehicle equidistant spacing
micro_simulation.jeep_system._space_jeeps_equidistantly()

print("\nExecuting microscopic simulation run loop...")
production_report = micro_simulation.run()

### Step 7: Analytical Post-Processing and Telemetry Reporting

Exports simulation logs and compiles operational telemetry metrics for your thesis.

In [ ]:
output_dir = Path("outputs/reports/")
output_dir.mkdir(parents=True, exist_ok=True)
production_report.export_report(out_dir=str(output_dir))

m = production_report.metrics

# Compute fleet headway variance
allocations = list(primary_chromosome.allocation.values())
mean_alloc = sum(allocations) / len(allocations) if allocations else 0.0
fleet_variance = sum((x - mean_alloc) ** 2 for x in allocations) / len(allocations) if allocations else 0.0

print("=== FINAL SYSTEM TELEMETRY ===")
print(f"Total Passenger Journeys Completed:        {m['completed_count']}")
print(f"Average Commute Duration (Ticks):          {m['mean_commute_time']:.2f}")
print(f"Unserved Demand / Market Starvation Count: {m['incomplete_count']}")
print(f"Fleet Headway Allocation Variance:         {fleet_variance:.4f}")